# About BERT


In [ ]:
!pip install datasets transformers tokenizers evaluate -q

## Немного теории

Важные даты:
- 2013 год - появление Word2Vec
- 2017 год - публикация "Attention is all you need" [1] с представлением Трансформеров и механизма внимания (но идея внимания сильно старше, чем эта статья)
- 2018 - 2019 год - представлен BERT [2] (в 2018 - на сайте архива, в 2019 - на NAACL)
- 2022 - настоящее время - активное развитие больших языковых моделей

**2013 - 2017 год: что здесь было?**

В отличие от предшественников, BERT (как представитель семейства Трансформеров) решает несколько проблем:
- Он позволяет получаеть контекстные эмбеддинги, что выгодно отличает его от word2vec & Co
- Внутри себя он умеет понимать, какие слова более важны друг для друга (за это отвечает механизм внимания), что позволяет в определенной степени решить проблемы с памятью, свойственные рекурентным моделям

Обучается BERT обычно на две задачи:
- Masked Language Modelling (MLM) - предсказание токена, закрытого маской
- Next Sentence Prediction (NSP) - определение, является ли второе предложение продолжением первого

Другие Трансформеры часто обучают на задачу Causal Language Modelling (CLM) - предсказание следующего токена по контексту

__Механизм внимания__

[Источник картинок](https://jalammar.github.io/illustrated-transformer/)

<img src='https://jalammar.github.io/images/t/transformer_self-attention_visualization.png'>

- Абстрактно: это часть модели, которая каждому входному токену ставит в соотвествие вектор, описывающий важность всех слов входной последовательночти
- Более конкретно: перемножение трех матриц (запрос, ключ и значение), полученных из статических эмбеддингов (да-да, они все еще с нами!)

__Целый трансформер__


<img src='https://jalammar.github.io/images/t/The_transformer_encoders_decoders.png'>

- Обычно энкодер + декодер: две половинки модели
- Но BERT - это только энкодер, а GPT (все его версии) - это только декодер. Разница, по большей части, в том, куда смотрит внимание: в энкодере внимание может смотреть на все предложение, так как он должен уметь его превратить в эмбеддинги; в декодере - только на предшествующие токены, так как декодер чаще всего про генерацию

|Encoder|Decoder|Encoder-Decoder|
|--|--|--|
|BERT|GPT|T5|
|RoBERTa|Qwen|Bart|
|ALBERT|DeepSeek|Pegasus|
|ELECTRA|LLaMA|ProphetNet|




Источники:
1. Vaswani A. et al. Attention is all you need //Advances in neural information processing systems. – 2017. – Т. 30.
2. Devlin J. et al. BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding //Proceedings of NAACL-HLT. – 2019. – С. 4171-4186.

[Очень хорошая статья про работу Трансформера](https://habr.com/ru/articles/486358/)

[На английском, но с крутыми картинками](https://jalammar.github.io/illustrated-transformer/)

## Hugging Face

[Hugging Face](https://huggingface.co/) - огромный агрегатор всего, связанного с DL моделями

Там есть:
- Датасеты для различных задач (NLP, CV, STT, TTS etc.)
- Модели, обученные на этих датасетах
- Множество связанных библиотек для работы с моделями

Если не знаете, какую модель использовать, то лучше начать с HF

Библиотеки, которые интересны нам, как лингвистам:
- datasets
- transformers
- tokenizers
- реже evaluate

Еще иногда полезно посмотреть [Tasks](https://huggingface.co/tasks), особенно в образовательных целях или если вы не знаете, как правильно реализовать обучение на конкретную задачу

И еще у них есть неплохой [курс по NLP](https://huggingface.co/learn/nlp-course/chapter0/1?fw=pt)

## Datasets

Cначала быстро посмотрим на датасеты. У них достаточно простой интерфейс, однако есть несколько нюансов.

В любом случае, сначала датасет надо прочитать. Сделать это можно несколькими способами, в том числе:
- прочитать датасет целиком
- прочитать конкретный сплит (причем здесь можно еще и срез по элементам указать)

In [ ]:
from datasets import load_dataset

Читаем целый датасет

In [ ]:
dataset = load_dataset("blinoff/kinopoisk")

In [ ]:
dataset

Читаем кусочек датасета

In [ ]:
small_dataset = load_dataset("blinoff/kinopoisk", split='train[10:20]')

In [ ]:
small_dataset

Самое очевидное, что можно дальше захотеть от датасета, это взять по нему конкретный срез (особенно, если изначально читали его целиком). Теоретически, датасет позволяет взаимодействие как со словарями и списками, однако тут есть одна беда: он возвращает словарь, а не датасет меньшего размера.

In [ ]:
dataset['train'][1:3]

Чтобы получить датасет меньшего размера, можно использовать `select`

In [ ]:
dataset['train'].select([1, 2])

Еще есть `filter`, который позволяет взять кусочек датасета по условию

In [ ]:
dataset.filter(lambda item: float(item['grade10']) > 9)

Еще удобная функция аналогична той, что есть в sklearn, и позволяет делить датасет на сплиты

In [ ]:
dataset['train'].train_test_split(test_size=0.1)

## Tokenizer

In [ ]:
import seaborn as sns
from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, AutoModel

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

Токенайзер для моделей вроде BERT также обучается, поэтому важно брать тот, который подходит к вашей модели. Обычно они идут парой под одним названием.

Чаще всего для BERT используется WordPiece токенайзер. Но чтобы понять, как он устроен, лучше сначала посмотреть на BPE (Byte Pair Encoding) токенайзер, суть которого заключаестя в токенизации на кусочки слов. Учится он следующим образом:

0. изначально словарь состоит из отдельных символов
1. считаем частотность сочетаний
2. потом самая частотная комбинация токенов помещается в словарь как отдельный токен
3. Повторяем пункты 1-2, пока размер словаря не станет заданного размера (это гиперпараметр, который задается при обучении)

Такое обучение позвоялет избежать проблемы с неизвестными словами: даже если это что-то совсем маргинальное, токенайзер просто в крайнем случае поделит на отдельные символы. Плюс, для совсем новых символов обычно есть специальный токен, обозначающий неизвестные символы (но чаще всего это редкая ситуация).

Теперь про WordPiece:
0. Начинаем также с отдельных символов. Но теперь символы в середине слова предваряются специальным префиксом (в BERT - `##`)
1. Объединяем символы, но теперь используем не просто частоту, а результат расчета по формуле:
\begin{align}
        score = \frac{freq(x_i, x_j)}{freq(x_i) * freq(x_j)}
\end{align}
Это позволяет чаще объединять пары, в которых кусочки редкие по отдельности.  

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

У токенайзера есть три (может, и больше) варианта использования:
- метод `tokenize`
- метод `encode`
- вызов токенайзера как callable объекта

In [ ]:
tokenizer.tokenize('Что ты здесь делаешь?')

In [ ]:
tokenizer.encode('Что ты здесь делаешь?')

In [ ]:
tokenizer.decode(tokenizer.encode('Что ты здесь делаешь?'))

In [ ]:
tokenizer('Что ты здесь делаешь?')

В контексте токенизации для дальнейшего использования при векторизации лучше всего использовать последний вариант. У токенайзера как функции есть множество параметров. Часть из них лучше/можно не трогать, но некоторые нужно знать:
- `return_tensors` - повзволяет выбрать, какой тип данных должен быть на выходе. Есть вариант `tf` (`tensorflow`), `pt` (`pytorch`) и  `np` (`numpy`). Если не указано, то вернет просто списки
- `padding` - задает, нужен ли и какой паддинг (дополнение) в инпуте. Умеет дополнять: до самой большой длины в инпуте или до максимальной длины, указанной в параметре
- `truncation` - нужно ли обрезать последовательность
- `max_length` - параметр максимальной длины последовательности


In [ ]:
text1 = 'Что ты здесь делаешь?'
text2 = 'Сижу на дереве.'

In [ ]:
tokenizer(text1, return_tensors='pt', max_length=16)

In [ ]:
tokenizer(text1, padding='max_length', truncation=True, return_tensors='pt',
          max_length=16)

А еще токенайзер умеет правильно обрабатывать пары текстов (полезно для задач типа NLI) или работать сразу со списком текстов

In [ ]:
tokenizer(text1, text2, padding='max_length', truncation=True, return_tensors='pt',
          max_length=16)

In [ ]:
tokenizer.decode(tokenizer.encode(text1, text2))

In [ ]:
tokenizer([text1, text2], padding='max_length', truncation=True, return_tensors='pt',
          max_length=16)

## Model

С моделью история та же, что и с токенайзером: ее сначала надо загрузить. Это может занять какое-то время, так как модель все-таки тяжелее.

In [ ]:
model = AutoModel.from_pretrained(
    "DeepPavlov/rubert-base-cased",
    attn_implementation="eager" # в других нельзя достать матрицы внимания
)

Дальше можно посмотреть, из чего она состоит

In [ ]:
model

Теперь посмотрим на то, как этим пользоваться:
1. Сначала токенизируем последовательность
2. Потом всё! полученное передаем в модель, вызывая ее как функцию

In [ ]:
toks = tokenizer(text1, padding='max_length', truncation=True, return_tensors='pt',
          max_length=16)
with torch.no_grad():
    model_output = model(**{k: v.to(model.device) for k, v in toks.items()},
                         output_attentions=True,
                         output_hidden_states=True)

Вообще, при большом желании, можно получить результат от работы только части модели, но для этого обычно нужна какая-то очень специфическая потребность.

Например, посмотрим на статические эмбеддинги из Берта. Теоретически, здесь можно изучать всякие интересные закономерности: как эти эмбеддинги связаны с тем, что в итоге оказывается в контексте (особенно в случае с задачей WSI) или работаю ли для них те же правила, что и для эмбеддингов из word2vec (пропорции и т.д.). Но важно, что здесь придется учесть привычку токенайзера делить слова на кусочки.

In [ ]:
model.embeddings.word_embeddings(toks['input_ids'])

Теперь посмотрим, что из себя представляет выдача всей модели

In [ ]:
type(model_output)

In [ ]:
model_output.keys()

In [ ]:
model_output.last_hidden_state, model_output.last_hidden_state.shape

In [ ]:
model_output.hidden_states[1], model_output.hidden_states[0].shape, len(model_output.hidden_states)

Отдельно можно посмотреть на матрицы внимания. И даже построить для них график

In [ ]:
model_output.attentions[0], model_output.attentions[0].shape, len(model_output.attentions)

In [ ]:
mask_example = toks["attention_mask"][0].bool()
attn_mean = torch.mean(model_output.attentions[0], dim=1).squeeze()
attn_masked = attn_mean[mask_example][:, mask_example]

tokens = tokenizer.convert_ids_to_tokens(toks["input_ids"][0])
tokens = [t for t, m in zip(tokens, toks["attention_mask"][0]) if m]

In [ ]:
sns.heatmap(attn_masked, xticklabels=tokens, yticklabels=tokens, cmap="viridis");

Еще одна полезная возможность (к сожалению, доступная большинству из нас только в колабе) - это перенос модели на гпу. Для этого нужно, чтобы в среде была настроена видеокарта.

In [ ]:
torch.cuda.is_available()

In [ ]:
device_cuda = torch.device("cuda")
device_cpu = torch.device("cpu")

In [ ]:
texts = list(dataset['train'].select(range(2))['content'])
toks = tokenizer(texts, padding='max_length', truncation=True, return_tensors='pt',
    max_length=256)

In [ ]:
model = model.to(device_cpu)

In [ ]:
%%timeit
with torch.no_grad():
    model_output = model(**{k: v.to(model.device) for k, v in toks.items()})

Важно, что для работы на гпу нужно перенести туда все, что будет использоваться при вычислениях: и модель, и результаты токенизации

In [ ]:
model = model.to(device_cuda)

In [ ]:
%%timeit
with torch.no_grad():
    model_output = model(**{k: v.to(model.device) for k, v in toks.items()})

## Hooks

У pytorch есть интересная механика - хуки. Они позволяют перехватывать и модифицировать данные внутри модели при прямом (_forward_) или обратном (_backward_) проходе.

Это часто гораздо удобнее для отладки или получения/изменения внутренних значений для пробинга, чем принты или сохранение в переменные.

In [ ]:
hidden_states_storage = []

def hook_fn(module, input, output):
    '''
    module - ссылка на модуль, на котором зарегестрирован хук
    input - входы модуля
    output -  выходы модуля
    Если сделать return в хуке, то он заменит output.
    '''
    # output.shape = [batch_size, seq_len, hidden_size]
    hidden_states_storage.append(output.detach())

In [ ]:
text1

In [ ]:
inputs = tokenizer(text1, return_tensors="pt")

Чтобы использовать хук, его надо зарегистрировать.

!Важно: если вы не сохраните никуда то, что вам возвращает регистратор хуков, то потом не сможете его удалить

In [ ]:
hooks = []
for i, layer in enumerate(model.encoder.layer):
    h = layer.register_forward_hook(hook_fn)
    hooks.append(h)

Используем и смотрим, что досталось

In [ ]:
with torch.no_grad():
    outputs = model(**{k: v.to(model.device) for k, v in inputs.items()})

In [ ]:
for i, hs in enumerate(hidden_states_storage):
    print(f"Layer {i} hidden state shape: {hs.shape}")
hs

Чтобы удалить хук, надо взять ту штуку, которую мы раньше удалили

In [ ]:
for h in hooks:
    h.remove()

## Trainer

Теперь давайте попробуем решить какую-нибудь задачу, используюя библиотеку transformers. Будем предсказывать, какую тональность имеет отзыв

In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer
import evaluate
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

Прочитаем датасет (чтобы не волноваться, что мы могли до этого что-то в нем сломать). Оставляем только 1000 строк, так как у нас не так много времени на обучение. На всем датасете код ниже будет работать около полутора часов

In [ ]:
dataset = load_dataset("blinoff/kinopoisk")
dataset = dataset['train'].select(range(1000))

Токенизируем весь датасет (так можно делать только с небольшими датасетами)

In [ ]:
def tokenize(example):
    return tokenizer(
        example["content"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(tokenize, batched=True)

Всякая полезная предобарботка:
- Переименовываем колонку в label, чтобы точно понимать, с чем работаем (ну и библиотека от нас ожидает такое название);
- Переводим лейблы в формат float, так как в датасете они исходно строки;
- Убираем лишнюю колонку с текстами;
- Настраиваем для нужных колонок формат, совместимый с pytorch, чтобы можно было обучать нейросети;
- Делим датасет на трейн и тест.


In [ ]:
def convert_label(example):
    example["label"] = float(example["label"])
    return example

dataset = dataset.rename_column("grade10", "label")
dataset = dataset.map(convert_label)
dataset = dataset.remove_columns(["content"])

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "label"]
)

dataset = dataset.train_test_split(test_size=0.1, seed=42)

Прочитаем модель еще раз, уже как основу для регрессии. Если убрать `model.config.problem_type = "regression"`, а в `num_labels=` поставить нужное чисто классов, то можно решать задачу классификации

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "DeepPavlov/rubert-base-cased",
    num_labels=1
)
model.config.problem_type = "regression"

Напишем те метрики, которые хотим считать. Закоменченная версия будет работать для более старых версий evaluate

In [ ]:
# rmse = evaluate.load("rmse")
# mae = evaluate.load("mae")
# r2 = evaluate.load("r2")

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     # regression logits.shape = (batch_size,1)
#     preds = logits.squeeze(-1)  # (batch_size,)
#     return {
#         "rmse": rmse.compute(predictions=preds, references=labels)["rmse"],
#         "mae": mae.compute(predictions=preds, references=labels)["mae"],
#         "r2": r2.compute(predictions=preds, references=labels)["r2"]
#     }

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.squeeze(-1)  # (batch_size,)

    mse = mean_squared_error(labels, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(labels, preds)
    r2 = r2_score(labels, preds)

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }

Настройки обучения

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    save_total_limit=1
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

Обучаем и проверяем

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

## PyTorch?

Если вдруг вы по какой-то причине испытывыете неприязнь к HF, то в pytorch есть интерфейс для загрузки предобученных моделей с HF. Однако весь интерфейс внутри такой же и зависимости тоже такие же. Только список моделей сильно меньше, плюс он почти не поддерживается и сейчас скорее не работает.

Зато, в отличие от HF, в pytorch есть возможность собирать полностью свои архитектуры, которые потом обучать. Причем это можно делать на разном уровне детализации:
- в pytorch есть как `nn.Transformer`, позволяющий просто перечислить кол-во слоев и получить модель,
- так и `nn.TransformerEncoder` с `nn.TransformerEncoderLayer` и `nn.TransformerDecoder` с `nn.TransformerDecoderLayer`, позволяющие собрать Трансформер по частям
- отдельно есть `nn.MultiheadAttention`, который реализует сам механизм внимания